In [1]:
!git clone https://github.com/salute-developers/GigaAM.git
%cd GigaAM
!pip install -q -e .

Cloning into 'GigaAM'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 61 (delta 11), reused 7 (delta 7), pack-reused 41 (from 1)
Receiving objects: 100% (61/61), 1.56 MiB | 6.81 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/kaggle/working/GigaAM
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 16.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.5/906.5 MB 1.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.8 MB/s et

In [11]:
!pip install -q speechbrain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 864.1/864.1 kB 12.8 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.1/739.1 kB 37.8 MB/s eta 0:00:00


In [12]:
!pip install -q jiwer

In [13]:
!pip install -q -U denoiser

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.8/49.8 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.5 MB/s eta 0:00:00


## Подготовка сетпапа для инференса GigaAM модели

In [3]:
import gigaam
import torch
onnx_dir = "onnx"
model_type = "ctc" # or "ctc"

asr_model = gigaam.load_model(
   model_type,
   fp16_encoder=True,
   use_flash=False,  # disable flash attention
)

asr_model.to_onnx(dir_path=onnx_dir)

100%|███████████████████████████████████████| 444M/444M [00:38<00:00, 12.1MiB/s]
/kaggle/working/GigaAM/gigaam/__init__.py:118: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
 

Succesfully ported onnx v2_ctc to onnx/v2_ctc.onnx.


In [4]:
from gigaam.onnx_utils import load_onnx_sessions, transcribe_sample

sessions = load_onnx_sessions(onnx_dir, model_type)

In [5]:
import os

os.environ["HF_TOKEN"] = ""


def get_transcription(audio_path):
    return transcribe_sample(audio_path, model_type, sessions)

In [15]:
from speechbrain.inference.separation import SepformerSeparation as separator
import torchaudio

## Инициализация модели разделения речи

In [16]:
model = separator.from_hparams(source="speechbrain/sepformer-wsj02mix", savedir='pretrained_models/}sepformer-wsj02mix', run_opts={"device": "cuda"})


hyperparams.yaml:   0%|          | 0.00/1.51k [00:00<?, ?B/s]

masknet.ckpt:   0%|          | 0.00/113M [00:00<?, ?B/s]

encoder.ckpt:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

decoder.ckpt:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/speechbrain/utils/checkpoints.py:200: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path, map_location=device

## Примеры разделения речи для английского языка

In [24]:
from IPython import display as disp

for i in range(1, 5):
    file_name = f"/kaggle/input/rus-two-voice-example/eng{i}.wav"
    wav, sr = torchaudio.load(file_name)
    est_sources = model.separate_file(file_name)
    disp.display(disp.Audio(wav.cpu(), rate=sr))
    disp.display(disp.Audio(est_sources[:, :, 0].cpu(), rate=8000))
    disp.display(disp.Audio(est_sources[:, :, 1].cpu(), rate=8000))

Resampling the audio from 48000 Hz to 8000 Hz


Resampling the audio from 48000 Hz to 8000 Hz


Resampling the audio from 48000 Hz to 8000 Hz


Resampling the audio from 48000 Hz to 8000 Hz


## Пример работы denoiser модели

In [25]:
from IPython import display as disp
import torch
import torchaudio
from denoiser import pretrained
from denoiser.dsp import convert_audio

In [26]:
denoiser_model = pretrained.dns64().cuda()


In [27]:
wav, sr = torchaudio.load('/kaggle/input/rus-two-voice-example/tel_call_2.wav')
wav = convert_audio(wav.cuda(), sr, denoiser_model.sample_rate, denoiser_model.chin)
with torch.no_grad():
    denoised = denoiser_model(wav[None])[0]
disp.display(disp.Audio(wav.data.cpu().numpy(), rate=denoiser_model.sample_rate))
disp.display(disp.Audio(denoised.data.cpu().numpy(), rate=denoiser_model.sample_rate))

In [28]:
def denoise_file(file_path, file_outout):
    wav, sr = torchaudio.load(file_path)
    wav = convert_audio(wav.cuda(), sr, denoiser_model.sample_rate, denoiser_model.chin)
    with torch.no_grad():
        denoised = denoiser_model(wav[None])[0]
    torchaudio.save(file_outout, denoised.detach().cpu(), denoiser_model.sample_rate)


## Функции для формирования файлов для теста

In [29]:
import random
import torch
import torchaudio
import os

# объединение файлов
def merge_wav_with_random_offset( 
    dataset_path: str,
    file1: str,
    file2: str,
    output_file: str,
    max_offset_coef: float = 2.0
):
    wav1, sr1 = torchaudio.load(os.path.join(dataset_path, file1))  # [C1, T1], sr1
    wav2, sr2 = torchaudio.load(os.path.join(dataset_path, file2))  # [C2, T2], sr2
    
    assert sr1 == sr2, "Обе дорожки должны иметь одинаковый sample_rate"
    sr = sr1
    if wav1.shape[1] > wav2.shape[1]:
        longer_audio = wav1
        shorter_audio = wav2
    else:
        longer_audio = wav2
        shorter_audio = wav1
    
    max_offset = int(longer_audio.shape[1] * max_offset_coef)

    offset = random.randint(0, max_offset)


    pad1 = longer_audio
    pad2 = torch.nn.functional.pad(shorter_audio, (offset, 0))

    len1 = pad1.shape[1]
    len2 = pad2.shape[1]
    max_len = max(len1, len2)
    pad1 = torch.nn.functional.pad(pad1, (0, max_len - len1))
    pad2 = torch.nn.functional.pad(pad2, (0, max_len - len2))

    merged = pad1 + pad2
    peak = merged.abs().max()
    if peak > 1.0:
        merged = merged / peak

    torchaudio.save(output_file, merged, sr)
    return offset

In [35]:
import pandas as pd
import os
from tqdm import tqdm

def fill_dict_record(idx, dataset_path, output_dict, row1, row2, offset):
    output_dict["id"].append(idx)
    output_dict["audio_path_1"].append(row1["audio_path"])
    output_dict["audio_path_2"].append(row2["audio_path"])
        
    output_dict["duration_1"].append(row1["duration"])
    output_dict["duration_2"].append(row2["duration"])

    with open(os.path.join(dataset_path, row1["txt_path"])) as f:
        output_dict["text_1"].append(f.read())

    with open(os.path.join(dataset_path, row2["txt_path"])) as f:
        output_dict["text_2"].append(f.read())
    
    output_dict["offset"].append(offset)
    

# Создание итогового датафрейма
def create_openstt_dataset(manifest_path, dataset_path, output_path, num_samples=50):
    random.seed(42)
    df = pd.read_csv(manifest_path, header=None, names=['audio_path', 'txt_path', 'duration'])
    df = df.sample(frac=1).reset_index(drop=True).iloc[:num_samples]
    output_dict = {"id": [],
        "audio_path_1": [], "audio_path_2": [],
                   "text_1": [], "text_2": [],
                   "duration_1": [], "duration_2": [],
                   "offset": [],}
    os.makedirs(output_path, exist_ok=True)

    for i in tqdm(range(0, len(df) - 1, 2)):
        if random.random() < 0.5:
            offset_coef = 0
        else:
            offset_coef = 2
        row1 = df.iloc[i]
        row2 = df.iloc[i + 1]
        offset = merge_wav_with_random_offset(dataset_path,
                                              row1["audio_path"], row2["audio_path"],
                                              os.path.join(output_path, f"{i // 2}.wav"),
                                              offset_coef)
        fill_dict_record(i // 2, dataset_path, output_dict, row1, row2, offset)
        
    out_df = pd.DataFrame.from_dict(output_dict)
    return out_df


    
        

In [34]:
youtube_df = create_openstt_dataset("/kaggle/input/openstt-youtube/public_youtube700_val.csv", "/kaggle/input/openstt-youtube/public_youtube700_val", "youtube_out", num_samples=50)

100%|██████████| 25/25 [00:01<00:00, 20.87it/s]


In [36]:
# Подсчет WER метрики (функция принимает пути к фалам и исходные тексты)
def calculate_audios_wer(path1, path2, ref1, ref2):
    hyp1 = get_transcription(path1)
    hyp2 = get_transcription(path2)

    w11 = calculate_wer(hyp1, ref1)
    w12 = calculate_wer(hyp1, ref2)
    w21 = calculate_wer(hyp2, ref1)
    w22 = calculate_wer(hyp2, ref2)

    sum_a = w11 + w22
    sum_b = w12 + w21

    if sum_a <= sum_b:
        return w11, w22, hyp1, hyp2
    else:
        return w21, w12, hyp2, hyp1

## Разделение файлов


In [37]:
import time 
def separate_files(audios_path, df, dataset_label):
    out1_dir = os.path.join(dataset_label, "out1")
    out2_dir = os.path.join(dataset_label, "out2")
    os.makedirs(out1_dir, exist_ok=True)
    os.makedirs(out2_dir, exist_ok=True)
    separation_times = []
    
    for idx, row in tqdm(df.iterrows()):
        file_path = os.path.join(audios_path, str(row["id"]) + ".wav")
        st = time.time()
        est_sources = model.separate_file(file_path)
        separation_times.append(time.time() - st)
        filename = (str(row["id"]) + ".wav")

        save_file_path_1 = os.path.join(str(out1_dir), f"{filename}")
        save_file_path_2 = os.path.join(str(out2_dir), f"{filename}")
        
        torchaudio.save(save_file_path_1, est_sources[:, :, 0].detach().cpu(), 8000)
        #denoise_file(save_file_path_1, save_file_path_1)
        torchaudio.save(save_file_path_2, est_sources[:, :, 1].detach().cpu(), 8000)
        #denoise_file(save_file_path_2, save_file_path_2)
    return separation_times

In [38]:
separate_files("youtube_out", youtube_df, "youtube_sep_out")

1it [00:00,  8.51it/s]

Resampling the audio from 16000 Hz to 8000 Hz
Resampling the audio from 16000 Hz to 8000 Hz


3it [00:00,  6.85it/s]

Resampling the audio from 16000 Hz to 8000 Hz
Resampling the audio from 16000 Hz to 8000 Hz


6it [00:00,  8.48it/s]

...

23it [00:03,  6.70it/s]

Resampling the audio from 16000 Hz to 8000 Hz
Resampling the audio from 16000 Hz to 8000 Hz


25it [00:03,  7.17it/s]

Resampling the audio from 16000 Hz to 8000 Hz


[0.04021477699279785,
 0.04017782211303711,
 0.04054832458496094,
 0.03910374641418457,
 0.03592681884765625,
 0.03327345848083496,
 0.03999733924865723,
 0.04040718078613281,
 0.0347590446472168,
 0.03764081001281738,
 0.03636908531188965,
 0.03878140449523926,
 0.03461503982543945,
 0.03903913497924805,
 0.03990006446838379,
 0.04152178764343262,
 0.03934979438781738,
 0.03386640548706055,
 0.03804492950439453,
 0.03937339782714844,
 0.03799748420715332,
 0.039008378982543945,
 0.03839278221130371,
 0.03809928894042969,
 0.033812761306762695]

# Подсчет метрик

## Функция подсчета WER метрики (для уже распознанного текста

In [39]:
from jiwer import wer, cer, Compose, ToLowerCase, RemovePunctuation, RemoveMultipleSpaces, Strip

transform = Compose([
    ToLowerCase(),
    RemovePunctuation(),
    RemoveMultipleSpaces(),
    Strip()
])
def calculate_wer(reference, hypothesis):
    reference_processed = transform(reference)
    hypothesis_processed = transform(hypothesis)
    if (not reference_processed) or (not hypothesis_processed):
        return 1.0
    error = wer(reference_processed, hypothesis_processed)
    return error

In [40]:
def evaluate_dataset(input_dir, df):
    in1_dir = os.path.join(input_dir, "out1")
    in2_dir = os.path.join(input_dir, "out2")
    eval_dict = {"wer_1": [], "wer_2": [], "recognized_text_1": [], "recognized_text_2": []}
    
    
    for idx, row in tqdm(df.iterrows()):
        filename_1 = os.path.join(in1_dir, str(idx) + ".wav")
        filename_2 = os.path.join(in2_dir, str(idx) + ".wav")
        wer_1, wer_2, recognized_text_1, recognized_text_2 = calculate_audios_wer(filename_1, filename_2, row["text_1"], row["text_2"])
        eval_dict["wer_1"].append(wer_1)
        eval_dict["wer_2"].append(wer_2)
        eval_dict["recognized_text_1"].append(recognized_text_1)
        eval_dict["recognized_text_2"].append(recognized_text_2)
    eval_df = pd.DataFrame.from_dict(eval_dict)
    df_merged = pd.concat([df, eval_df], axis=1)
    return df_merged

        

In [41]:
# Датасет с аудио с ютуба
youtube_eval_df = evaluate_dataset("youtube_sep_out", youtube_df)

25it [01:34,  3.76s/it]


In [48]:
youtube_eval_df

,id,audio_path_1,audio_path_2,text_1,text_2,duration_1,duration_2,offset,wer_1,wer_2,recognized_text_1,recognized_text_2
0,0,public_youtube700_val/9/48/3870c5ec36a2.wav,public_youtube700_val/4/81/f3b257d7f420.wav,конечно было цель\n,резиновой пищей\n,1.69,1.60,1639,3.000000,0.500000,иа,резиновой пи
1,1,public_youtube700_val/9/04/5d8a73703f95.wav,public_youtube700_val/3/fb/c2e85b6d3392.wav,место касающиеся того что речь идёт только о м...,мы используем сложный признак то он всё равно\n,3.64,2.09,32098,0.100000,0.428571,место касающиеся того что речь идет только о м...,мы исхольуем сложный признак он все равно
2,2,public_youtube700_val/c/ae/13a51a8a7bc2.wav,public_youtube700_val/9/75/d59b3fcf0e3c.wav,и думаю что этот напиток сегодня ааа пополнит ...,у людей подвалы\n,3.87,0.97,0,0.125000,0.500000,и думаю что этот напиток сегодня пополнит вашу,у людей подвалыя пополнит
3,3,public_youtube700_val/d/92/31fa996ccd3a.wav,public_youtube700_val/e/a4/2c2399bb0c6d.wav,мы это с вами здание ремонтировали мы его пост...,после неё\n,2.48,0.81,71482,0.000000,0.500000,мы это с вами здание ремонтировали мы его пост...,после нее
4,4,public_youtube700_val/b/29/b67df97c3b23.wav,public_youtube700_val/a/6e/f4f15527e569.wav,не выгоняйте мы больше\n,это как\n,1.42,0.47,0,0.250000,1.000000,не выгодяете мы больше,не погодяйте мы больше
5,5,public_youtube700_val/3/91/b8dd0fba42a8.wav,public_youtube700_val/1/6e/11120f0706b0.wav,постоянно\n,и взбалтываем\n,0.48,0.75,0,1.000000,2.000000,ида,остоян
6,6,public_youtube700_val/a/d2/5eeb1469253d.wav,public_youtube700_val/b/45/e7f2060a48c7.wav,очень счастлив\n,сначала даже не понимаешь почему у этой пещеры...,0.96,5.12,0,1.000000,0.111111,оче тои не понимаешь почему у этой пещеры тако...,сначала даже не понимаешь почему этой пещеры т...
7,7,public_youtube700_val/1/b9/dad1d9df192c.wav,public_youtube700_val/a/3b/1cf0672fb04b.wav,а этот столик из бревна выброшенного на берег ...,что же произошло\n,4.43,0.85,109974,0.100000,0.000000,а это столик из бревна выброшенного на берег в...,что же произошло
8,8,public_youtube700_val/a/0c/900f579a4883.wav,public_youtube700_val/f/30/98ed3469041e.wav,вот и вы заснете вот и всё\n,не обязательно висеть\n,1.26,0.92,0,0.833333,0.750000,тоо нать нед вот и все,не выза снимет вытаще
9,9,public_youtube700_val/a/d3/f1c226fae376.wav,public_youtube700_val/7/d2/850ff25938d8.wav,а вообще я занимаюсь этим с шестнадцати лет из...,это уровень собственно инфраструктурный\n,2.45,1.98,851,0.000000,1.000000,а вообще я занимаюсь этим с шестнадцати лет из...,а вообще занимтеь шиноните


In [42]:
youtube_eval_df["wer_1"].mean()

0.6140989010989011

In [43]:
youtube_eval_df["wer_2"].mean()

0.564946608946609

In [44]:
# Датасет с аудио из телефонных разговоров.
phone_df = create_openstt_dataset("/kaggle/input/opentts-phone-calls/asr_calls_2_val.csv", "/kaggle/input/opentts-phone-calls/asr_calls_2_val/asr_calls_2_val", "phone_out")

100%|██████████| 25/25 [00:01<00:00, 20.52it/s]


In [45]:
separate_files("phone_out", phone_df, "phone_sep_out")

1it [00:00,  4.98it/s]

Resampling the audio from 16000 Hz to 8000 Hz
Resampling the audio from 16000 Hz to 8000 Hz


2it [00:00,  5.94it/s]

Resampling the audio from 16000 Hz to 8000 Hz


3it [00:00,  4.86it/s]

...

Resampling the audio from 16000 Hz to 8000 Hz
Resampling the audio from 16000 Hz to 8000 Hz


24it [00:04,  4.83it/s]

Resampling the audio from 16000 Hz to 8000 Hz
Resampling the audio from 16000 Hz to 8000 Hz


25it [00:04,  5.31it/s]


[0.04387021064758301,
 0.03901052474975586,
 0.042386770248413086,
 0.04712486267089844,
 0.04088592529296875,
 0.058200836181640625,
 0.035085201263427734,
 0.03911709785461426,
 0.0391087532043457,
 0.03990888595581055,
 0.04186534881591797,
 0.0390172004699707,
 0.0388188362121582,
 0.04034709930419922,
 0.038167715072631836,
 0.04021286964416504,
 0.03839588165283203,
 0.030567407608032227,
 0.04044389724731445,
 0.04345083236694336,
 0.04211115837097168,
 0.03401756286621094,
 0.042230844497680664,
 0.03980708122253418,
 0.03885912895202637]

In [46]:
phone_eval_df = evaluate_dataset("phone_sep_out", phone_df)

25it [01:55,  4.63s/it]


In [47]:
phone_eval_df

,id,audio_path_1,audio_path_2,text_1,text_2,duration_1,duration_2,offset,wer_1,wer_2,recognized_text_1,recognized_text_2
0,0,asr_calls_2_val/5/ea/d3b016c61a38.wav,asr_calls_2_val/4/d8/291270ce3707.wav,а он уже там давно\n,я сам говорю мне звонили изз октябрьского ра э...,1.15,5.56,6556,0.769231,0.777778,да он уже там нас никак не касается я же вам у...,я он уже там давзвонили изз краснооктябрьского...
1,1,asr_calls_2_val/7/f8/0670993ef29d.wav,asr_calls_2_val/5/c1/a9b8dab3ae9f.wav,как у тебя дела там\n,алло дарова петух ты чё мне в контакте пишешь ...,0.84,3.44,32098,0.571429,0.625000,петух ты чтокак у тебя дела такиевещи,алло здорово пит в контакте пишешь такие вещи
2,2,asr_calls_2_val/2/6a/67b1dc024279.wav,asr_calls_2_val/6/bc/96f9760051f6.wav,я ничего не хочу никем быть яяя тот кто я есть...,ака двести в руках держал\n,7.65,1.85,0,0.391304,1.000000,я ничего не хочу ниеить я то что есть короче п...,не кажтьесть я то то и есть короче послушай ты...
3,3,asr_calls_2_val/6/93/c68080251b13.wav,asr_calls_2_val/a/b0/15a00e8778ff.wav,вот и я тоже спрашиваю куда прибыть\n,мне просто мне это уже всё надоело вот вы звон...,1.86,7.87,233879,0.852941,0.214286,мну просто мне это уже все надоело вот вы звон...,ну просто мне это уже все надоело вот вы звони...
4,4,asr_calls_2_val/e/b3/b88b463fa4e7.wav,asr_calls_2_val/5/47/b2711e476aa3.wav,погоду самая клёвая мне каж мне очень нравится...,здрасти\n,3.27,0.68,77397,0.333333,1.000000,погода самая клевая мне кажется мне очень нрав...,погода самая клевая мне кажется мне очень нрав...
5,5,asr_calls_2_val/5/9b/6d0be15979dc.wav,asr_calls_2_val/2/ac/f4ac8eb0b5f7.wav,вы сами мне всё рассказывали чё вы сюда звонит...,с чего это вдруг\n,3.78,0.96,0,0.181818,0.833333,вы сами мне все рассказывали что вы сюда звони...,вы чего это вы все рассказывали чего вы сюда з...
6,6,asr_calls_2_val/a/05/8fa7fd2fd230.wav,asr_calls_2_val/7/74/6e71bc83d2b8.wav,запрудник конечно мы коренные все\n,вокзал как я туда приеду\n,1.84,1.65,0,1.333333,1.000000,и конечно кривнысе,вопрос кавя оттуда приеду
7,7,asr_calls_2_val/3/49/fd8bd20b6719.wav,asr_calls_2_val/5/60/6b034a4ff8d5.wav,что за комитет он нас никак не касается я ж ва...,и ссышь приехать ты\n,2.93,1.17,3478,0.600000,2.000000,слоба комитета нас никак не касается же вам уж...,не звонил
8,8,asr_calls_2_val/9/b2/5d88b6dcb137.wav,asr_calls_2_val/9/4d/54ad2cc7ae1d.wav,я шёл из военкомата\n,ебало твоё блядь вонючая\n,1.32,1.79,46925,0.750000,1.000000,ало твое блять вонючее я се с военкомата,бало твое блять вонючее я сорч военкомата
9,9,asr_calls_2_val/5/46/24c02241b5c8.wav,asr_calls_2_val/3/d4/f37615dcbd7b.wav,приём приём\n,у вас получится значит своего\n,0.66,1.74,35713,1.000000,0.285714,у вас получится значит своего прием принял,у вас получится значит своего прием принял
